# 📊 MixBhasha — Full Analytics & GitHub/Drive Export (v5)

**Runs on your existing `MixBhasha_V5_seed42` Google Drive experiment folder.** It does **not** train models or evaluate held-out test labels. It reads already-saved validation outputs, creates detailed figures/tables for **each trained model**, and produces three possible archives:

1. **GitHub-ready source ZIP** — safe code, tests, docs, aggregate validation charts. **No raw data or model weights.**
2. **All graphs and metrics ZIP** — visualizations, aggregate HTML/Markdown report, numeric CSV tables for every model.
3. Optional **PRIVATE selected-checkpoints ZIP** or **PRIVATE full-backup ZIP** — large files **never push to GitHub**.

✅ Works even if you no longer have the original Python training source in Colab: this notebook embeds the clean v5 project source and reporting tools.

**Research precautions:** all charts are **validation**, not test. The cyberbullying annotation review was a non-random sample of model mistakes; its percentages are not dataset-wide rates. The sample's alleged mislabels must not be treated as confirmed errors.


In [ ]:
# 1) Mount Google Drive. Only set the two paths below.
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab; skipping Drive mount. Set ROOT to an existing local folder.')

ROOT = Path('/content/drive/MyDrive/MixBhasha_V5_seed42')
EXPORT_DIR = Path('/content/drive/MyDrive/MixBhasha_Exports')

# Optional: PRIVATE backups of trained weights. Off by default because models are huge.
MAKE_PRIVATE_SELECTED_MODELS = False
SELECTED_MODELS = ['sentiment_raw:hing_roberta_mixed', 'cyberbullying7:tfidf_svm']
MAKE_PRIVATE_FULL_BACKUP = False       # All raw data/private annotations/models/results, large; NEVER publish.
INCLUDE_TRAINER_CHECKPOINTS = False  # Extremely large. Existing best/final weights included without this.

# Paired bootstrap resamples (validation-only). Raise to 2000 for a more stable exploratory CI.
BOOTSTRAP_REPEATS = 700

assert ROOT.exists(), f'Experiment ROOT not found: {ROOT}. Check the Drive folder path.'
assert (ROOT / 'models').is_dir(), f'Missing models/ under {ROOT}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Reading experiment:', ROOT)
print('Saving exports to:', EXPORT_DIR)


## 2 — Restore project utilities bundled inside this notebook

The next code cell reconstructs a **source-only** project folder under `/content` (never from the user's private data folder). It does not modify any saved model checkpoints or training results.


In [ ]:
import base64, io, zipfile, shutil, sys
from pathlib import Path

_BOOTSTRAP_BASE64 = """..."""
CODE_HOME = Path('/content/MixBhasha_Release_Source')
CODE_HOME.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_BOOTSTRAP_BASE64))) as archive:
    archive.extractall(CODE_HOME)
SOURCE_DIR = CODE_HOME / 'MixBhasha'
assert (SOURCE_DIR / 'src' / 'mixbhasha_core.py').exists()
assert (SOURCE_DIR / 'tools' / 'generate_report.py').exists()
assert (SOURCE_DIR / 'tools' / 'export_project.py').exists()
sys.path.insert(0, str(SOURCE_DIR / 'tools'))
print('Bundled source restored to:', SOURCE_DIR)
print('Project tools ready. Your Drive models remain untouched.')


## 3 — Generate comprehensive validation plots and CSV tables

Per task:
- all-model leaderboard (accuracy, macro-F1, weighted-F1, balanced accuracy)
- class counts (validation only) and per-class F1/precision/recall tables
- raw and normalized confusion matrices **for each model**
- one-vs-rest ROC and precision–recall curves **for each model**
- per-model confidence/reliability diagrams and diagnostic Brier/ECE
- all-model per-class heatmap, prediction agreement heatmap
- 700-replicate paired bootstrap comparison of the top two models
- training/evaluation loss and F1 curves **only if checkpoint histories exist**
- optional **aggregate-only** manual error reason breakdown if the private reviewed CSV exists

The plots and CSVs are saved in the same Drive experiment folder under `reports/public_validation/`. No original text is exported.


In [ ]:
# Install only if the runtime lacks the graphing and metrics dependencies.
# %pip -q install numpy pandas matplotlib scikit-learn  # uncomment if needed
from generate_report import create_report

REPORT_DIR = ROOT / 'reports' / 'public_validation'
report = create_report(ROOT, REPORT_DIR, bootstrap=BOOTSTRAP_REPEATS)
if not report:
    raise RuntimeError('No saved validation artifacts were found. Check ROOT/models/*/*/validation_probabilities.npz')
print('HTML dashboard:', REPORT_DIR / 'REPORT.html')
for task, record in report.items():
    print(f"{task}: {len(record['models'])} models, n={record['n_validation']}, top={record['best_model']}, val macro-F1={record['best_validation_macro_f1']:.4f}")


## 4 — Display results within Colab

Renders CSV leaderboards and a few representative figures. Your Drive archive has **every model**'s figures, not just these previews.


In [ ]:
from IPython.display import display, Image, HTML
import pandas as pd
for task in report:
    print('\nTASK:', task)
    display(pd.read_csv(REPORT_DIR/task/'leaderboard.csv').round(4))
    display(pd.read_csv(REPORT_DIR/task/'class_f1_comparison.csv').round(4))
    display(Image(filename=str(REPORT_DIR/'charts'/task/'model_metrics_comparison.png')))
    display(Image(filename=str(REPORT_DIR/'charts'/task/'class_f1_heatmap.png')))


## 5 — ZIP everything for download from Google Drive

Creates a **small public GitHub ZIP** and **separate graphs/metrics ZIP**. You can switch on *private* checkpoint backups in Cell 1 — model backups may be multiple GB and require enough Google Drive free storage. They are intentionally separate because raw data and checkpoints should not automatically be pushed to GitHub.


In [ ]:
from export_project import make_repo_zip, make_report_zip, make_selected_models_backup, make_full_private_backup

REPO_ZIP = EXPORT_DIR / 'MixBhasha_GitHub_Ready.zip'
REPORT_ZIP = EXPORT_DIR / 'MixBhasha_All_Graphs_And_Metrics.zip'
make_repo_zip(SOURCE_DIR, REPO_ZIP, REPORT_DIR)
make_report_zip(REPORT_DIR, REPORT_ZIP)

private_exports=[]
if MAKE_PRIVATE_SELECTED_MODELS:
    p = EXPORT_DIR / 'MixBhasha_SELECTED_MODELS_PRIVATE.zip'
    make_selected_models_backup(ROOT, p, SELECTED_MODELS)
    private_exports.append(p)
if MAKE_PRIVATE_FULL_BACKUP:
    p = EXPORT_DIR / 'MixBhasha_FULL_BACKUP_PRIVATE.zip'
    make_full_private_backup(ROOT, p, INCLUDE_TRAINER_CHECKPOINTS)
    private_exports.append(p)

print('\n✅ Open Google Drive → My Drive → MixBhasha_Exports and download:')
for f in [REPO_ZIP, REPORT_ZIP] + private_exports:
    print(f'  {f.name}: {f.stat().st_size / 1024**2:.2f} MiB')


## 6 — Optional browser download of the small ZIPs

The archives are already in Google Drive. Use Google Drive's **Download** button for large files. The direct `files.download` option below may be unreliable for files over ~100 MB.


In [ ]:
# Optional. Uncomment only if you want an immediate download in Colab.
# from google.colab import files
# files.download(str(REPO_ZIP))
# files.download(str(REPORT_ZIP))

print('Ready:')
print(REPO_ZIP)
print(REPORT_ZIP)
print('Unzip MixBhasha_GitHub_Ready.zip, review README.md, then push that SOURCE folder to a GitHub repository.')
print('DO NOT push *_PRIVATE.zip or ROOT/data, ROOT/models, ROOT/private_analysis.')


## 7 — Verify the generated archives

Optional integrity check. Both ZIPs should open normally. This does not re-train or evaluate models.


In [ ]:
with zipfile.ZipFile(REPO_ZIP) as z:
    print('GitHub ZIP entries:', len(z.namelist()), 'integrity:', 'OK' if z.testzip() is None else 'FAILED')
with zipfile.ZipFile(REPORT_ZIP) as z:
    print('Report ZIP entries:', len(z.namelist()), 'integrity:', 'OK' if z.testzip() is None else 'FAILED')
print('Done ✅')
